<a href="https://colab.research.google.com/github/Vermont-Complex-Systems/storywrangler/blob/main/notebooks/registering_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U storywrangler

In [ ]:
from storywrangler import Storywrangler

# Registering dataset

In this notebook, we will need an API key for registering data. See [docs](https://storywrangler.uvm.edu/authentication) for details. We added a key to the colab using the `Secrets` tab in the colab editor.

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('storywrangler')

In [ ]:
client=Storywrangler(api_key=API_KEY)

In [ ]:
# client.users.whoami() # to verify you are logged in

## Custom endpoints - minimal payload

Say that we have this data we want to register:

```
┌────────────────────────────┐  
│         *.parquet          │  
│                            │  
│ country          varchar   │  
│ date             timestamp │  
│ avg_happs        double    │  
│ hedo_total_count bigint    │  
│ avg_power        double    │  
│ avg_danger       double    │  
│ avg_structure    double    │  
│ pds_total_count  bigint    │  
│ total_count      bigint    │  
└────────────────────────────┘  
```

We will want to query based on `country`, with `date` being a time-parsable columns. The other columns can be returned as such. Here's the simplest payload to register the data; it says where the can be found and where it comes from, what does it looks like, who owns it, and specify that the date column is a `time_dimension` in `transform`. That's it. It might not seems much, but this is enough for people to have a sense of it.

In [ ]:
from storywrangler.registry import DatasetCreate

In [ ]:
# ?DatasetCreate

In [ ]:
# ?client.registry.register

In [ ]:
# using DatasetCreate is nice because you
# get to see the fields and their type.
payload = DatasetCreate(
    catalog = "vcsi",
    domain = "wikimedia",
    dataset_id = "semantic-timeseries-test",
    description = "Daily lexicon-scored time series per country: pageview-weighted labMT happiness (avg_happs) and ousiometric power/danger/structure scores (PDS), with the word-count denominators behind each average. One row per country per day; 'All' is the pre-aggregated global corpus.",
    data_location = "/netfiles/wikimedia_snapshots/wiki_hedonometer/aggregate/hedonometer_aggregate.parquet",
    data_format = "parquet",
    transform = {"time_dimension": "date"},
    ownership = {"owner_group": "vcsi", "contact": "compstorylab@uvm.edu", "status": "active"},
    lineage = {
        "sources": {"main": {"enwiki": "https://dumps.wikimedia.org/other/enterprise_html/"}},
        "derived_from": ["wikimedia/ngrams"],
        "repo": "https://github.com/Vermont-Complex-Systems/wikipedia-parsing"
    }
)

In [ ]:
client.registry.register(payload)

  semantic-timeseries-test registered successfully!


In [ ]:
test = client.dataset("wikimedia", "semantic-timeseries-test")

In [ ]:
test.meta # it now exists in the registry; but it only worked because
          # hedonometer_aggregate.parquet is actually reachable from the virtual machine.

{'catalog': 'vcsi',
 'domain': 'wikimedia',
 'dataset_id': 'semantic-timeseries-test',
 'version': 'latest',
 'schema_version': '0.1.0',
 'data_location': '/netfiles/wikimedia_snapshots/wiki_hedonometer/aggregate/hedonometer_aggregate.parquet',
 'data_format': 'parquet',
 'description': "Daily lexicon-scored time series per country: pageview-weighted labMT happiness (avg_happs) and ousiometric power/danger/structure scores (PDS), with the word-count denominators behind each average. One row per country per day; 'All' is the pre-aggregated global corpus.",
 'manifest': {'availability': {'min': '2024-10-01 00:00:00',
   'max': '2026-07-18 00:00:00'}},
 'data_schema': {'country': 'VARCHAR',
  'date': 'TIMESTAMP',
  'avg_happs': 'DOUBLE',
  'hedo_total_count': 'BIGINT',
  'avg_power': 'DOUBLE',
  'avg_danger': 'DOUBLE',
  'avg_structure': 'DOUBLE',
  'pds_total_count': 'BIGINT',
  'total_count': 'BIGINT'},
 'entity_mapping': None,
 'endpoint_schema': None,
 'transform': {'time_dimension': 

Some extra fields were introspecting upon registration

In [ ]:
test.availability # under the hood, we use the dimension to get time range

{'min': '2024-10-01 00:00:00', 'max': '2026-07-18 00:00:00'}

In [ ]:
test.meta['data_schema'] # the metadata of the files are introspected to access the schema

{'country': 'VARCHAR',
 'date': 'TIMESTAMP',
 'avg_happs': 'DOUBLE',
 'hedo_total_count': 'BIGINT',
 'avg_power': 'DOUBLE',
 'avg_danger': 'DOUBLE',
 'avg_structure': 'DOUBLE',
 'pds_total_count': 'BIGINT',
 'total_count': 'BIGINT'}

Now, the metadata was registered. But it isn't doing much on it's own yet. It is not a standardized endpoints, so we can't use it out of the box (more on that below). To serve custom datasets such as this one, we made the choice that there needs to be some code review happening. In this case, the dataset is very minimal so the Storywrangler maintainers would add this endpoint to the `wikimedia.py` router:

```python
@router.get("/semantic-timeseries", openapi_extra={"x-dataset": "semantic-timeseries"})
async def semantic_timeseries(
    country: str = Query("United States", description="Country name as stored in the data (e.g. 'United States'), or 'All' for the global pageview-weighted corpus"),
    db: AsyncSession = Depends(get_session),
):
    """Daily lexicon-scored time series for one country's pageview-weighted corpus.

    Returns the full history: one entry per day with the labMT happiness score
    (avg_happs), ousiometric power/danger/structure scores, and the
    pageview-weighted word-count denominators behind each average.
    """
    dataset_obj = await get_latest_entry(db, "wikimedia", "semantic-timeseries")
    if not dataset_obj:
        raise HTTPException(status_code=404, detail="'wikimedia/semantic-timeseries' dataset not found")

    def _query():
        # handle_query_error translates DuckDB failures into API errors:
        # timeout → 504, missing data file → 404, anything else → a 500
        # that doesn't leak filesystem paths.
        with handle_query_error("wikimedia/semantic-timeseries"):
            # timed_connect hands this request its own cursor on the shared
            # connection (parquet metadata stays cached across requests) and
            # interrupts the query after 120s — the timeout the 504 refers to.
            with get_duckdb_client().timed_connect() as conn:
                cur = conn.execute(
                    f"""
                    SELECT * REPLACE (strftime(date, '%Y-%m-%d') AS date)
                    FROM read_parquet('{dataset_obj.data_location}')
                    WHERE country = ?
                    ORDER BY date
                    """,
                    [country],
                )
                columns = [d[0] for d in cur.description]
                return [dict(zip(columns, row)) for row in cur.fetchall()]

    # DuckDB is blocking: run_blocking moves the query to a bounded worker
    # pool so a slow read doesn't stall every other request on the API.
    return await run_blocking(_query)

```

Each endpoint is composed of the following parts, which we show here just for transparency. In most cases, we (or our Agents) will be implementing this:
- Access the dataset object using `get_latest_entry` to get started. Under the hood, we are serving most recent release of a dataset (more on versioning later)
- Define a `duckdb` query:
  - We have utils to handle all kinds of query errors, i.e. `handle_query_error`
  - Connect to `duckdb` client
  - `time_connect()` so that we time out the query if this is taking too long.
  - Define the actual `_query` that will be served, return everything in this case.
  - We make sure all endpoints are not stepping on each other's toes using `await run_blocking(_query)`

It is far from ideal to have such hurdle be part of the registration, but at the same time we do want to know all the data that will be hosted on the Storywrangler platform. This is not a bug, and Storywrangler is not meant to scale indefinitely. The idea is to have a community-based platform that can be replicated across communities but still require cares to ensure the quality of the data we are nurturing on the long term.

## Making use of Storywrangler axes

In here, we show the `entity` and `transform` system. Say we have this dataset:
```
┌───────────┬─────────┬────────┬───────┬───────────────┐
│   types   │   sex   │ counts │ year  │      geo      │
│  varchar  │ varchar │ int32  │ int32 │    varchar    │
├───────────┼─────────┼────────┼───────┼───────────────┤
│ Mary      │ F       │   7065 │  1880 │ united_states │
│ Anna      │ F       │   2604 │  1880 │ united_states │
│ Emma      │ F       │   2003 │  1880 │ united_states │
│ Elizabeth │ F       │   1939 │  1880 │ united_states │
│ Minnie    │ F       │   1746 │  1880 │ united_states │
└───────────┴─────────┴────────┴───────┴───────────────┘
```
First, lets use the `wikidata` identifiers to encode geography. We do that as `yaml` file;

In [ ]:
import yaml

entities = {
    "united_states": {
        "entity_id": "wikidata:Q30",
        "entity_name": "United States",
        "entity_ids": [
            "iso:US",
            "local:babynames:united_states",
        ],
    },
    "quebec": {
        "entity_id": "wikidata:Q176",
        "entity_name": "Quebec",
        "entity_ids": [
            "iso:CA-QC",
            "local:babynames:quebec",
        ],
    },
}

with open("entities.yaml", "w", encoding="utf-8") as f:
    yaml.dump(entities, f, sort_keys=False, default_flow_style=False, allow_unicode=True)

Although the entity system is optional, entities are useful facilitate future joins across datasets.

In [ ]:
def get_entities() -> list[dict]:
    """Load entity mappings from config/entities.yaml."""
    with open("./entities.yaml") as f:
        mappings = yaml.safe_load(f)
    return [{"local_id": local_id, **mapping} for local_id, mapping in mappings.items()]

In [ ]:
DatasetCreate(
    catalog = "vcsi",
    dataset_id = "ngrams",
    domain = "babynames",
    data_location = [
        '/netfiles/babynames/united_states.parquet',
        '/netfiles/babynames/quebec.parquet'
    ],
    data_format = "parquet",
    description = "Baby names by popularity, year, and location with entity mappings",
    entity_mapping = {
        "local_id_column": "geo",
    },
    entities = get_entities(),
    endpoint_schema = {
        "type": "types-counts",
    },
    transform = {
        "time_dimension": "year",
        "filter_dimensions": ["sex"],
    },
    ownership = {
        "owner_group": "vcsi",
        "contact": "compstorylab@uvm.edu",
        "storage_risk": "institutional",
    },
    lineage = {
            "sources": {"geo": {
                'quebec': [
                    'https://www.donneesquebec.ca/recherche/dataset/93d640ec-d059-4768-b7ed-388604b278aa/resource/039539f5-af55-4d8f-9010-ca718e45c2a5/download/g_1980-2024_masque.csv',
                    'https://www.donneesquebec.ca/recherche/dataset/13db2583-427a-4e5f-b679-8532d3df571f/resource/bf77b504-54b9-4db8-be53-b92156175c12/download/f_1980-2024_masquee.csv'
                    ],
                'united_states': 'https://www.ssa.gov/oact/babynames/names.zip'
            }},
            "derived_from": [],
            "repo": "https://github.com/Vermont-Complex-Systems/babynames",
        }
)

Notes:
- `data_location` is now a list of parquet files
- `entity_mapping` specify the column that contains the entities
- `endpoint_schema`: we declare that dataset of type `types-counts`, making it available to use with `allotaxonometer` instrument out of the box.
- `year` is our time dimension, with `sex` being declared as `filter_dimensions`.

In [ ]:
bb = client.dataset("babynames", "ngrams")

In [ ]:
bb.filters # note that by registering as filters, we are then injecting them as values for docs.

{'sex': {'default': None, 'valid': ['F', 'M']}}

In [ ]:
bb.adapter # entities are available as "adapter"

[{'local_id': 'united_states',
  'entity_id': 'wikidata:Q30',
  'entity_name': 'United States',
  'entity_ids': ['iso:US', 'local:babynames:united_states']},
 {'local_id': 'quebec',
  'entity_id': 'wikidata:Q176',
  'entity_name': 'Quebec',
  'entity_ids': ['iso:CA-QC', 'local:babynames:quebec']}]

we can now call `allotax` directly on the dataset, even without registering a new endpoint:

In [ ]:
result = bb.allotax(entity="united_states", dates=1968, dates2=2018, sex="F", alpha="Inf")
result["wordshift"][:5]

[{'type': 'Lisa (1 ⇋ 1628.5)', 'metric': -0.05161095608631549},
 {'type': 'Liam (11967 ⇋ 1)', 'metric': 0.05161095608631549},
 {'type': 'Michelle (2 ⇋ 556)', 'metric': -0.025805478043157744},
 {'type': 'Emma (426.5 ⇋ 2)', 'metric': 0.025805478043157744},
 {'type': 'Kimberly (3 ⇋ 406)', 'metric': -0.01720365202877183}]

Notes:
- Since we declared `geo` as entities, then we use the `entity` argument to filter. Similar with `year` being our time dimension, we use the `dates` conventional argument to filter it. In both cases, we can use `entity2` or `dates2` to compare a pair of systems.
- Under the hood, the `top-ngrams` endpoint assume that we can filter time ranges using strings. Duckdb is taking care of the aggregation.

To see the underlying data, we still need to add an addtional `top-ngrams` endpoint. But we are working to make it available as with the instrument ones:

In [ ]:
bb.top_ngrams(
    dates="2000,2002",
    dates2="2002,2004",
    sex="F",
    entity="united_states",
    limit = 5
).df()

,types,counts,system
0,Emily,75487.0,2000-2002
1,Madison,63907.0,2000-2002
2,Hannah,62637.0,2000-2002
3,Ashley,49875.0,2000-2002
4,Alexis,49671.0,2000-2002
5,Emily,75203.0,2002-2004
6,Madison,62602.0,2002-2004
7,Emma,60896.0,2002-2004
8,Hannah,52078.0,2002-2004
9,Olivia,46892.0,2002-2004
